# BoxBunny — Master Runner Notebook

This notebook contains all commands needed to build, run, test, and manage the BoxBunny boxing training robot system.

**Demo Users:**
| User | Password | Level | Sessions | Rank |
|------|----------|-------|----------|------|
| alex | boxing123 | Beginner | 8 | Novice |
| maria | boxing123 | Intermediate | 35 | Fighter |
| jake | boxing123 | Advanced | 120 | Champion |
| sarah | coaching123 | Coach | 3 coaching | — |

---
## 0. Environment Setup
Run this cell first to set up the environment.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
# Source ROS 2 and workspace
echo "=== Environment Setup ==="
source /opt/ros/humble/setup.bash
if [ -f install/setup.bash ]; then
    source install/setup.bash
    echo "Workspace overlay sourced"
fi
echo "ROS_DISTRO: $ROS_DISTRO"
echo "Python: $(python3 --version)"
echo "Working dir: $(pwd)"
echo ""
echo "=== Package Status ==="
ros2 pkg list 2>/dev/null | grep boxbunny || echo "(packages not built yet — run Build cell first)"

---
## 1. Build Workspace
Builds all 4 ROS 2 packages: boxbunny_msgs, boxbunny_core, boxbunny_gui, boxbunny_dashboard

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash
echo "=== Building ROS 2 Workspace ==="
colcon build --symlink-install 2>&1
echo ""
echo "=== Build Complete ==="
source install/setup.bash
echo "Packages built:"
ros2 pkg list 2>/dev/null | grep boxbunny

---
## 2. Verify Messages
Check that all custom ROS 2 messages and services are built correctly.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "=== Custom Messages ==="
ros2 interface list | grep boxbunny_msgs | head -30
echo ""
echo "=== Sample Message: ConfirmedPunch ==="
ros2 interface show boxbunny_msgs/msg/ConfirmedPunch
echo ""
echo "=== Sample Service: StartSession ==="
ros2 interface show boxbunny_msgs/srv/StartSession

---
## 3. Run Tests
Runs the full pytest suite (171 tests — no hardware needed).

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
python3 -m pytest tests/ -v --tb=short 2>&1

---
## 4. Seed Demo Data
Creates demo users with realistic training history. Safe to re-run (idempotent).

In [ ]:
%%bash
set +e
python3 tools/demo_data_seeder.py

---
## 5. Launch System — Development Mode
Launches all ROS nodes + IMU simulator + GUI. Use this for development/testing.

**Note:** This launches in the background. Use the Stop cell below to shut down.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Launching BoxBunny in dev mode..."
echo "(IMU simulator will open in a separate window)"
echo ""
ros2 launch boxbunny_core boxbunny_dev.launch.py &
LAUNCH_PID=$!
echo "Launch PID: $LAUNCH_PID"
echo "Run the 'Stop System' cell to shut down"
echo $LAUNCH_PID > /tmp/boxbunny_launch.pid
sleep 5
echo ""
echo "=== Active ROS Nodes ==="
ros2 node list 2>/dev/null || echo "(nodes still starting...)"

---
## 6. Launch System — Full Production Mode
Launches everything including real hardware connections.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Launching BoxBunny in FULL mode..."
ros2 launch boxbunny_core boxbunny_full.launch.py &
LAUNCH_PID=$!
echo "Launch PID: $LAUNCH_PID"
echo $LAUNCH_PID > /tmp/boxbunny_launch.pid
sleep 5
ros2 node list 2>/dev/null

---
## 7. Stop System
Shuts down all BoxBunny ROS nodes.

In [ ]:
%%bash
set +e
export QT_QPA_PLATFORM_PLUGIN_PATH=$(python3 -c "import PySide6; print(PySide6.__path__[0])")/Qt/plugins/platforms
if [ -f /tmp/boxbunny_launch.pid ]; then
    PID=$(cat /tmp/boxbunny_launch.pid)
    kill $PID 2>/dev/null && echo "Stopped launch process $PID" || echo "Process already stopped"
fi
# Kill any remaining boxbunny nodes
pkill -f 'boxbunny_core' 2>/dev/null
pkill -f 'boxbunny_gui' 2>/dev/null
pkill -f 'boxbunny_dashboard' 2>/dev/null
pkill -f 'imu_simulator' 2>/dev/null
echo "All BoxBunny processes stopped"

---
## 8. Launch Individual Components
Run specific subsystems for isolated testing.

### 8a. IMU Simulator Only

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Starting IMU Simulator..."
python3 tools/imu_simulator.py &
sleep 2
echo "IMU Simulator running. Click pads to publish ROS topics."

### 8b. GUI Only

In [ ]:
%%bash
set +e
export QT_QPA_PLATFORM_PLUGIN_PATH=$(python3 -c "import PySide6; print(PySide6.__path__[0])")/Qt/plugins/platforms
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
export QT_QPA_PLATFORM=xcb
echo "Starting BoxBunny GUI..."
python3 -m boxbunny_gui.app &
sleep 2
echo "GUI launched"

### 8c. Dashboard Server Only
Starts the dashboard with a **public tunnel URL** you can open on your phone from anywhere.
**Interrupt the cell (stop button) to shut down the server.**

In [ ]:
import subprocess, socket, time, sys, os

os.chdir('/home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws')
ws = '/home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws'

# Kill old
os.system('fuser -k 8080/tcp 2>/dev/null')
os.system('pkill -f localhost.run 2>/dev/null')
time.sleep(1)

# Get IP
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    s.connect(('8.8.8.8', 80))
    ip = s.getsockname()[0]
    s.close()
except: ip = 'localhost'

# Start server as subprocess that stays attached (not -m which can fork)
env = os.environ.copy()
env['PYTHONPATH'] = f'{ws}/src/boxbunny_dashboard:{ws}/src/boxbunny_core:{ws}/install/boxbunny_msgs/local/lib/python3.10/dist-packages:' + env.get('PYTHONPATH', '')

server = subprocess.Popen(
    [sys.executable, '-c',
     'import uvicorn; from boxbunny_dashboard.server import create_app; uvicorn.run(create_app(), host="0.0.0.0", port=8080, log_level="warning")'],
    env=env
)
time.sleep(2)

if server.poll() is not None:
    print('ERROR: Server failed to start. Check dependencies.')
else:
    print(f'Server running (PID {server.pid})')
    print(f'Local: http://{ip}:8080')
print()

# Tunnel
tunnel = subprocess.Popen(
    ['ssh', '-o', 'StrictHostKeyChecking=no', '-o', 'ServerAliveInterval=30',
     '-R', '80:localhost:8080', 'nokey@localhost.run'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
tunnel_url = None
deadline = time.time() + 15
while time.time() < deadline:
    line = tunnel.stdout.readline().decode('utf-8', errors='ignore')
    if not line: break
    if '.lhr.life' in line:
        for w in line.split():
            if 'https://' in w:
                tunnel_url = w.strip().rstrip(',')
                break
        if tunnel_url: break

url = tunnel_url or f'http://{ip}:8080'
if tunnel_url:
    print(f'PUBLIC URL: {tunnel_url}')
else:
    print(f'LOCAL: http://{ip}:8080')
print('Logins: alex/boxing123 | maria/boxing123 | jake/boxing123 | sarah/coaching123')
print()

# QR
try:
    import qrcode
    from IPython.display import display, Image as IPImage
    from io import BytesIO
    qr = qrcode.QRCode(version=1, box_size=10, border=2)
    qr.add_data(url)
    qr.make(fit=True)
    img = qr.make_image(fill_color='white', back_color='#0A0A0A')
    buf = BytesIO()
    img.save(buf, format='PNG')
    buf.seek(0)
    print('Scan with your phone:')
    display(IPImage(data=buf.read(), width=350))
except:
    print(f'Open: {url}')

print()
print('=' * 50)
print('RUNNING — Press STOP to shut down')
print('=' * 50)

# Block on server process — uvicorn.run() keeps the process alive
# so server.wait() will block until interrupted
try:
    server.wait()
except KeyboardInterrupt:
    pass
finally:
    server.terminate()
    tunnel.terminate()
    try: server.wait(timeout=3)
    except: server.kill()
    try: tunnel.wait(timeout=3)
    except: tunnel.kill()
    os.system('fuser -k 8080/tcp 2>/dev/null')
    print('\nServer and tunnel stopped.')


### 8d. Headless Nodes Only (no GUI)

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Launching headless nodes..."
ros2 launch boxbunny_core headless.launch.py &
sleep 3
ros2 node list 2>/dev/null

---
## 9. Monitor ROS Topics
View live data flowing through the system.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "=== Active ROS Topics ==="
ros2 topic list 2>/dev/null | grep boxbunny | sort
echo ""
echo "=== Active Nodes ==="
ros2 node list 2>/dev/null | grep -v _ros2cli | sort
echo ""
echo "=== Topic Hz (punch confirmed) ==="
timeout 3 ros2 topic hz /boxbunny/punch/confirmed 2>/dev/null || echo "(no data yet)"

### 9a. Echo a specific topic (5 messages)

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
# Change topic name as needed:
TOPIC="/boxbunny/imu/nav_event"
echo "Listening to $TOPIC (5 messages, 10s timeout)..."
timeout 10 ros2 topic echo $TOPIC --once 2>/dev/null || echo "(no messages received — is a publisher running?)"

---
## 10. Database Inspection
View demo user data in the databases.

In [ ]:
import sqlite3, json

# Main database
conn = sqlite3.connect('data/boxbunny_main.db')
conn.row_factory = sqlite3.Row

print("=== Users ===")
for row in conn.execute("SELECT id, username, display_name, user_type, level, last_login FROM users"):
    print(f"  {dict(row)}")

print("\n=== Presets ===")
for row in conn.execute("SELECT user_id, name, preset_type, is_favorite, use_count FROM presets LIMIT 10"):
    print(f"  {dict(row)}")

print("\n=== Coaching Sessions ===")
for row in conn.execute("SELECT * FROM coaching_sessions"):
    print(f"  {dict(row)}")

print("\n=== Guest Sessions ===")
for row in conn.execute("SELECT * FROM guest_sessions"):
    print(f"  {dict(row)}")

conn.close()

In [ ]:
import sqlite3, json

# Per-user database — change username as needed
USERNAME = "maria"  # Try: alex, maria, jake

db_path = f'data/users/{USERNAME}/boxbunny.db'
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

print(f"=== {USERNAME}'s Training Sessions (last 10) ===")
for row in conn.execute("SELECT session_id, mode, difficulty, rounds_completed, is_complete FROM training_sessions ORDER BY started_at DESC LIMIT 10"):
    print(f"  {dict(row)}")

print(f"\n=== {USERNAME}'s XP & Rank ===")
for row in conn.execute("SELECT * FROM user_xp"):
    print(f"  {dict(row)}")

print(f"\n=== {USERNAME}'s Streak ===")
for row in conn.execute("SELECT * FROM streaks"):
    print(f"  {dict(row)}")

print(f"\n=== {USERNAME}'s Achievements ===")
for row in conn.execute("SELECT * FROM achievements"):
    print(f"  {dict(row)}")

print(f"\n=== {USERNAME}'s Personal Records ===")
for row in conn.execute("SELECT * FROM personal_records"):
    print(f"  {dict(row)}")

conn.close()

---
## 11. Model Verification
Check that all ML models are present and loadable.

In [ ]:
import os, sys
sys.path.insert(0, '.')

models = {
    'CV Action Model (.pth)': 'action_prediction/model/best_model.pth',
    'CV Action Model (.onnx)': 'action_prediction/model/best_model.onnx',
    'CV Action Model (.trt)': 'action_prediction/model/best_model.trt',
    'YOLO Pose Model': 'action_prediction/model/yolo26n-pose.pt',
    'LLM (Qwen2.5-3B)': 'models/llm/qwen2.5-3b-instruct-q4_k_m.gguf',
}

print("=== Model Status ===")
for name, path in models.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"  OK  {name}: {size_mb:.1f} MB")
    else:
        print(f"  MISSING  {name}: {path}")

# Test CV model loading
print("\n=== CV Model Load Test ===")
try:
    import torch
    ckpt = torch.load('action_prediction/model/best_model.pth', map_location='cpu', weights_only=False)
    config = ckpt.get('config', {})
    print(f"  Model loaded successfully")
    print(f"  Classes: {ckpt.get('label_names', 'default 8-class')}")
    print(f"  Config keys: {list(config.keys())[:10]}")
except Exception as e:
    print(f"  Load failed: {e}")

---
## 12. LLM Test
Test the local LLM model (Qwen2.5-3B).

In [ ]:
import time

MODEL_PATH = 'models/llm/qwen2.5-3b-instruct-q4_k_m.gguf'

try:
    from llama_cpp import Llama
    print("Loading LLM (this takes 10-30 seconds)...")
    start = time.time()
    llm = Llama(model_path=MODEL_PATH, n_gpu_layers=-1, n_ctx=512, verbose=False)
    load_time = time.time() - start
    print(f"Model loaded in {load_time:.1f}s")
    
    # Generate a coaching tip
    print("\nGenerating coaching tip...")
    start = time.time()
    result = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": "You are BoxBunny AI Coach, an expert boxing trainer. Keep responses under 2 sentences."},
            {"role": "user", "content": "I just did a training session: 87 punches, mostly jabs and crosses, accuracy 72%. Give me a quick tip."},
        ],
        max_tokens=80,
        temperature=0.7,
    )
    gen_time = time.time() - start
    tip = result['choices'][0]['message']['content']
    print(f"AI Coach says ({gen_time:.1f}s): {tip}")
    
    del llm  # Free GPU memory
except ImportError:
    print("llama-cpp-python not installed. Run: pip install llama-cpp-python")
except Exception as e:
    print(f"LLM test failed: {e}")

---
## 13. Camera Test
Test the RealSense D435i camera connection.

In [ ]:
try:
    import pyrealsense2 as rs
    ctx = rs.context()
    devices = ctx.query_devices()
    if len(devices) > 0:
        dev = devices[0]
        print(f"Camera found: {dev.get_info(rs.camera_info.name)}")
        print(f"Serial: {dev.get_info(rs.camera_info.serial_number)}")
        print(f"Firmware: {dev.get_info(rs.camera_info.firmware_version)}")
    else:
        print("No RealSense camera detected (connect D435i or use recorded data)")
except ImportError:
    print("pyrealsense2 not installed")
except Exception as e:
    print(f"Camera check failed: {e}")

---
## 14. Action Prediction — Standalone Test
Test the CV model inference without ROS (uses camera directly).

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Running action prediction standalone test..."
echo "(Press Ctrl+C or close the window to stop)"
python3 action_prediction/run.py \
    --checkpoint action_prediction/model/best_model.pth \
    --pose-weights action_prediction/model/yolo26n-pose.pt \
    2>&1 || echo "(camera not available — connect D435i to run this test)"

---
## 15. Dashboard Server Test
Start the phone dashboard and test API endpoints.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
# Start dashboard in background
python3 -m boxbunny_dashboard.server &
sleep 3

echo "=== Testing API ==="
echo ""
echo "--- Health Check ---"
curl -s http://localhost:8080/api/health | python3 -m json.tool 2>/dev/null || echo "(server not responding)"

echo ""
echo "--- Login as Alex ---"
TOKEN=$(curl -s -X POST http://localhost:8080/api/auth/login \
    -H 'Content-Type: application/json' \
    -d '{"username": "alex", "password": "boxing123"}' | python3 -c "import sys,json; print(json.load(sys.stdin).get('token',''))" 2>/dev/null)
echo "Token: ${TOKEN:0:20}..."

if [ -n "$TOKEN" ]; then
    echo ""
    echo "--- Alex's Session History ---"
    curl -s http://localhost:8080/api/sessions/history \
        -H "Authorization: Bearer $TOKEN" | python3 -m json.tool 2>/dev/null | head -30

    echo ""
    echo "--- Alex's Gamification Profile ---"
    curl -s http://localhost:8080/api/gamification/profile \
        -H "Authorization: Bearer $TOKEN" | python3 -m json.tool 2>/dev/null
fi

# Stop dashboard
pkill -f 'boxbunny_dashboard.server' 2>/dev/null
echo ""
echo "Dashboard test complete"

---
## 16. Build Vue Frontend
Rebuild the phone dashboard Vue 3 SPA.

In [ ]:
%%bash
set +e
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
cd src/boxbunny_dashboard/frontend
echo "Node: $(node --version), npm: $(npm --version)"
echo ""
echo "=== Installing dependencies ==="
npm install 2>&1 | tail -3
echo ""
echo "=== Building Vue SPA ==="
npm run build 2>&1
echo ""
echo "=== Output ==="
ls -lh ../static/dist/ 2>/dev/null | head -5

---
## 17. Download Models
Download LLM model and check all model files.

In [ ]:
%%bash
set +e
bash scripts/download_models.sh

---
## 18. Full Setup (First Time)
One-command bootstrap for a fresh system. Installs deps, builds, downloads, seeds.

In [ ]:
%%bash
set +e
bash scripts/setup.sh 2>&1

---
## 19. Hardware Check
Quick status check of all hardware components.

In [ ]:
import os, sys

checks = [
    ('RealSense Camera', lambda: __import__('pyrealsense2').context().query_devices().size() > 0),
    ('PyTorch', lambda: __import__('torch').__version__),
    ('CUDA Available', lambda: __import__('torch').cuda.is_available()),
    ('PySide6', lambda: __import__('PySide6').__version__),
    ('FastAPI', lambda: __import__('fastapi').__version__),
    ('MediaPipe', lambda: __import__('mediapipe').__version__),
    ('llama-cpp-python', lambda: __import__('llama_cpp').__version__ if hasattr(__import__('llama_cpp'), '__version__') else 'installed'),
    ('bcrypt', lambda: __import__('bcrypt').__version__),
    ('qrcode', lambda: 'installed' if __import__('qrcode') else ''),
    ('CV Model', lambda: os.path.exists('action_prediction/model/best_model.pth')),
    ('YOLO Model', lambda: os.path.exists('action_prediction/model/yolo26n-pose.pt')),
    ('LLM Model', lambda: os.path.exists('models/llm/qwen2.5-3b-instruct-q4_k_m.gguf')),
    ('Main DB', lambda: os.path.exists('data/boxbunny_main.db')),
]

print("=== Hardware & Dependency Check ===")
for name, check_fn in checks:
    try:
        result = check_fn()
        if isinstance(result, bool):
            status = 'OK' if result else 'MISSING'
        else:
            status = f'OK ({result})'
    except Exception as e:
        status = f'MISSING ({type(e).__name__})'
    symbol = '+' if 'OK' in status else '-'
    print(f"  [{symbol}] {name}: {status}")

---
## 20. Project Statistics

In [ ]:
%%bash
set +e
export QT_QPA_PLATFORM_PLUGIN_PATH=$(python3 -c "import PySide6; print(PySide6.__path__[0])")/Qt/plugins/platforms
echo "=== BoxBunny Project Stats ==="
echo "Python files: $(find . -name '*.py' -not -path './_archive/*' -not -path './build/*' -not -path './install/*' -not -path '*/__pycache__/*' | wc -l)"
echo "Vue/JS files: $(find . \( -name '*.vue' -o -name '*.js' \) -not -path '*/node_modules/*' -not -path './_archive/*' -not -path './build/*' | wc -l)"
echo "ROS messages: $(find src/boxbunny_msgs -name '*.msg' | wc -l)"
echo "ROS services: $(find src/boxbunny_msgs -name '*.srv' | wc -l)"
echo "GUI widgets:  $(find src/boxbunny_gui/boxbunny_gui/widgets -name '*.py' -not -name '__init__.py' | wc -l)"
echo "GUI pages:    $(find src/boxbunny_gui/boxbunny_gui/pages -name '*.py' -not -name '__init__.py' | wc -l)"
echo "ROS nodes:    $(find src/boxbunny_core/boxbunny_core -name '*_node.py' -o -name '*_manager.py' -o -name '*_engine.py' -o -name '*_processor.py' | wc -l)"
echo "API endpoints: $(find src/boxbunny_dashboard/boxbunny_dashboard/api -name '*.py' -not -name '__init__.py' | wc -l)"
echo "Knowledge docs: $(find data/boxing_knowledge -type f | wc -l)"
echo "Test files:    $(find tests -name 'test_*.py' | wc -l)"
echo "Config files:  $(find config -type f | wc -l)"
echo "Launch files:  $(find src -name '*.launch.py' | wc -l)"

---
## 21. Sound Test
Play each sound effect to verify audio output. The first cell lists all available sounds; the second plays them all in sequence.

### 21a. List Available Sounds
Inventory of all `.wav` assets with file sizes.

In [ ]:
import os, sys
sys.path.insert(0, 'src/boxbunny_gui')

sounds_dir = 'src/boxbunny_gui/assets/sounds'
print("=== Available Sound Effects ===")
print(f"Directory: {os.path.abspath(sounds_dir)}\n")

total_size = 0
for f in sorted(os.listdir(sounds_dir)):
    if f.endswith('.wav'):
        full_path = os.path.join(sounds_dir, f)
        size = os.path.getsize(full_path)
        total_size += size
        # Friendly name from filename
        label = f.replace('.wav', '').replace('_', ' ').title()
        print(f"  {label:.<30} {f:>25}  ({size:,} bytes)")

print(f"\n  Total: {len([f for f in os.listdir(sounds_dir) if f.endswith('.wav')])} files, {total_size:,} bytes")

# Play a single sample sound
from IPython.display import Audio, display
sample = os.path.join(sounds_dir, 'bell_start.wav')
print(f"\nPlaying sample: bell_start.wav")
print("(You should hear the round-start bell below)")
display(Audio(sample, autoplay=True))

### 21b. Play All Sounds in Sequence
Listen to every sound effect back-to-back with labels. Each sound plays as an embedded audio widget.

In [ ]:
import os
from IPython.display import Audio, display, HTML

sounds_dir = 'src/boxbunny_gui/assets/sounds'

# Description of when each sound is used in the app
sound_descriptions = {
    'bell_start.wav':        'Rings at the start of each round',
    'bell_end.wav':          'Rings when the round ends',
    'button_click.wav':      'Plays on any GUI button press',
    'coach_notification.wav': 'Plays when the AI coach has feedback',
    'countdown_beep.wav':    'Short beep during 3-2-1 countdown',
    'countdown_go.wav':      'Plays on "GO!" after countdown',
    'impact.wav':            'Triggered on confirmed punch impact',
    'reaction_stimulus.wav': 'Stimulus flash sound in reaction drill',
    'rest_start.wav':        'Signals the beginning of a rest period',
    'session_complete.wav':  'Celebratory sound when session finishes',
}

print("=== Full Sound Suite Playback ===")
print("Press the play button on each widget to hear the sound.\n")

for f in sorted(os.listdir(sounds_dir)):
    if not f.endswith('.wav'):
        continue
    full_path = os.path.join(sounds_dir, f)
    label = f.replace('.wav', '').replace('_', ' ').title()
    desc = sound_descriptions.get(f, '')
    display(HTML(
        f'<div style="background:#1A1A1A;padding:8px 12px;border-radius:8px;margin:4px 0;'
        f'font-family:monospace;color:#E0E0E0">'
        f'<b style="color:#00E676">{label}</b>'
        f'<span style="color:#9E9E9E;margin-left:12px">{desc}</span>'
        f'</div>'
    ))
    display(Audio(full_path, autoplay=False))

print("\nAll sounds loaded. Click play on each to test audio output.")

---
## 22. Dashboard with QR Code
Generate a scannable QR code for the phone dashboard, then start the server so you can connect from your phone.

**Prerequisite:** Run the *Seed Demo Data* cell (Section 4) first so users exist.

### 22a. Generate QR Code
Creates a QR code image that links to the dashboard URL on this device's local IP. Scan it with your phone (same Wi-Fi network required).

In [ ]:
import socket, io
import qrcode
from IPython.display import display, Image as IPImage, HTML

# Determine the device's LAN IP address (not 127.0.0.1)
def get_local_ip():
    """Get the LAN IP by briefly connecting to an external address."""
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.settimeout(0.5)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
        return ip
    except Exception:
        return socket.gethostbyname(socket.gethostname())

ip = get_local_ip()
port = 8080
url = f"http://{ip}:{port}"

# Generate QR code with BoxBunny styling
qr = qrcode.QRCode(version=1, box_size=10, border=2)
qr.add_data(url)
qr.make(fit=True)
img = qr.make_image(fill_color="white", back_color="#0D0D0D")

# Save to buffer and display
buf = io.BytesIO()
img.save(buf, format='PNG')
buf.seek(0)

display(HTML(f"""
<div style="background:#0D0D0D;padding:24px;border-radius:16px;text-align:center;
            max-width:400px;margin:0 auto;font-family:sans-serif">
    <h2 style="color:#00E676;margin:0 0 8px 0">BoxBunny Dashboard</h2>
    <p style="color:#E0E0E0;margin:0 0 4px 0">Scan with your phone camera to open</p>
    <p style="color:#9E9E9E;margin:0 0 16px 0;font-size:13px">
        Your phone must be on the same Wi-Fi network as this device
    </p>
</div>
"""))
display(IPImage(data=buf.read()))
display(HTML(f"""
<div style="text-align:center;font-family:monospace;margin-top:8px">
    <p style="color:#E0E0E0;font-size:18px"><b>{url}</b></p>
    <p style="color:#9E9E9E;font-size:13px">
        Login: <b>alex</b> / boxing123 &nbsp;|&nbsp; <b>maria</b> / boxing123 &nbsp;|&nbsp; <b>jake</b> / boxing123
    </p>
</div>
"""))

### 22b. Start Dashboard Server
Launches the dashboard server in the background. After running this cell, scan the QR code above to access the dashboard from your phone.

In [ ]:
%%bash
set +e
source /opt/ros/humble/setup.bash && source install/setup.bash

# Kill any existing dashboard process
pkill -f 'boxbunny_dashboard.server' 2>/dev/null
sleep 1

echo "=== Starting Dashboard Server ==="
python3 -m boxbunny_dashboard.server &
SERVER_PID=$!
echo $SERVER_PID > /tmp/boxbunny_dashboard.pid
sleep 3

# Verify it is running
if kill -0 $SERVER_PID 2>/dev/null; then
    echo "Dashboard server started (PID: $SERVER_PID)"
    echo ""
    echo "Open your phone browser and scan the QR code above, or visit:"
    echo "  http://$(hostname -I | awk '{print $1}'):8080"
    echo ""
    echo "To stop: run 'pkill -f boxbunny_dashboard.server' or use the Stop System cell"
else
    echo "ERROR: Dashboard server failed to start. Check logs above."
fi

---
## 23. Gesture Control Debug
Launch the gesture recognition node with a live camera debug window. This uses MediaPipe hand tracking to detect hand gestures for touchless GUI navigation.

**Requires:** RealSense D435i camera connected. Press Ctrl+C in the terminal (or interrupt the kernel) to stop.

In [ ]:
%%bash
set +e
source /opt/ros/humble/setup.bash && source install/setup.bash

echo "=== Gesture Control Debug Mode ==="
echo ""
echo "A camera window will open showing the live feed with hand landmarks."
echo ""
echo "Supported gestures:"
echo "  Open palm ........... Select / Enter"
echo "  Thumbs up ........... Confirm"
echo "  Peace sign .......... Go back"
echo "  Swipe left/right .... Navigate between options"
echo ""
echo "Hold each gesture for ~0.5 seconds to trigger the action."
echo "Press Ctrl+C or close the window to stop."
echo ""

ros2 run boxbunny_core gesture_node \
    --ros-args -p enabled:=true -p hold_duration_s:=0.5 2>&1 \
    || echo "(gesture node exited — is the camera connected?)"

---
## 24. IMU Simulator Visual Test
Launch the IMU pad simulator and monitor the nav events it publishes. Click the pad buttons in the simulator window to simulate punches and verify ROS message flow.

**No hardware required** -- the simulator replaces the physical IMU pads for testing.

In [ ]:
%%bash
set +e
source /opt/ros/humble/setup.bash && source install/setup.bash

echo "=== IMU Simulator Test ==="
echo ""
echo "A simulator window will open with clickable pad buttons."
echo ""
echo "Pad mapping for GUI navigation:"
echo "  LEFT pad  .... Previous option"
echo "  RIGHT pad .... Next option"
echo "  CENTRE pad ... Select / Enter"
echo "  HEAD pad ..... Go back"
echo ""
echo "Force levels (shown by colour):"
echo "  Light (green) | Medium (orange) | Hard (red)"
echo ""
echo "Click the pads and watch for nav events below."
echo "The monitor will run for 30 seconds, then stop automatically."
echo ""

# Start the simulator GUI in the background
python3 tools/imu_simulator.py &
SIM_PID=$!
echo "Simulator started (PID: $SIM_PID)"
sleep 2

echo ""
echo "--- Monitoring /boxbunny/imu/nav_event (30s timeout) ---"
echo "(Click pads in the simulator to see events appear here)"
echo ""
timeout 30 ros2 topic echo /boxbunny/imu/nav_event 2>/dev/null \
    || echo "(no events received -- click the pads in the simulator window)"

echo ""
echo "Stopping simulator..."
kill $SIM_PID 2>/dev/null
echo "Done."

---
## 25. CV Model Live Test
Run the action prediction model with the live camera feed. The window will show:
- YOLO pose skeleton overlay on your body
- Predicted punch type and confidence score
- Voxel activity heatmap

**Requires:** RealSense D435i camera + CV model files (see Section 11). Throw punches at the camera to see predictions update in real time. Press `q` or close the window to stop.

In [ ]:
%%bash
set +e
source /opt/ros/humble/setup.bash && source install/setup.bash

echo "=== Action Prediction Live Test ==="
echo ""
echo "A window will open showing the camera feed with overlays:"
echo "  - Pose skeleton (YOLO keypoints drawn on your body)"
echo "  - Predicted action label + confidence percentage"
echo "  - Voxel activity visualization"
echo ""
echo "Stand about 1.5-2m from the camera and throw punches!"
echo "Expected labels: jab, cross, hook_lead, hook_rear,"
echo "                 uppercut_lead, uppercut_rear, block, idle"
echo ""
echo "Press 'q' in the video window to stop."
echo ""

python3 action_prediction/run.py \
    --checkpoint action_prediction/model/best_model.pth \
    --pose-weights action_prediction/model/yolo26n-pose.pt \
    2>&1 || echo "(failed -- is the D435i camera connected and are models downloaded?)"

---
## 26. Developer Mode Visual Test
Launch the GUI with the developer overlay enabled. The overlay appears in the bottom-right corner and shows:
- 4 pad rectangles (HEAD, LEFT, CENTRE, RIGHT) that flash in colour on impacts
- 2 arm rectangles (L ARM, R ARM) that flash on strikes
- Current CV prediction with confidence bar

Start the IMU simulator in another terminal (or run the simulator cell above first) to see pads light up when you click them.

In [ ]:
%%bash
set +e
source /opt/ros/humble/setup.bash && source install/setup.bash
export QT_QPA_PLATFORM=xcb

echo "=== Developer Mode Test ==="
echo ""
echo "The GUI will open with the developer overlay in the bottom-right corner."
echo "It shows:"
echo "  - 4 pad rectangles (HEAD, LEFT, CENTRE, RIGHT) that flash on impacts"
echo "  - 2 arm rectangles (L ARM, R ARM) that flash on strikes"
echo "  - Current CV prediction with confidence bar"
echo ""
echo "Starting IMU simulator in background for pad testing..."

# Start IMU simulator in background so you can click pads
python3 tools/imu_simulator.py &
SIM_PID=$!
echo "IMU Simulator PID: $SIM_PID"
sleep 1

echo ""
echo "Starting GUI with developer mode enabled..."
echo "Close the GUI window to stop."
echo ""

# Launch the GUI and immediately enable developer mode
python3 -c "
import sys, os
sys.path.insert(0, 'src/boxbunny_gui')
os.environ['QT_QPA_PLATFORM'] = 'xcb'

from boxbunny_gui.app import BoxBunnyApp
app = BoxBunnyApp()
app.set_developer_mode(True)
exit_code = app.run()
sys.exit(exit_code)
" 2>&1

# Cleanup
kill $SIM_PID 2>/dev/null
echo "Stopped IMU simulator."

---
## 27. View Demo User Dashboards
Renders styled profile cards for each demo user, pulling live data from their SQLite databases. This is a quick visual check that the seeded data looks correct.

**Prerequisite:** Run *Seed Demo Data* (Section 4) first.

In [ ]:
import sqlite3, json, os
from IPython.display import HTML, display

demo_users = ['alex', 'maria', 'jake']
cards_html = ""

for username in demo_users:
    db_path = f'data/users/{username}/boxbunny.db'
    if not os.path.exists(db_path):
        print(f"Database not found for {username} -- run Section 4 (Seed Demo Data) first")
        continue

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    xp = dict(conn.execute("SELECT * FROM user_xp WHERE id=1").fetchone())
    streak = dict(conn.execute("SELECT * FROM streaks WHERE id=1").fetchone())
    sessions = conn.execute("SELECT COUNT(*) as c FROM training_sessions").fetchone()['c']
    achievements = conn.execute("SELECT COUNT(*) as c FROM achievements").fetchone()['c']

    # Recent personal records
    records = [dict(r) for r in conn.execute("SELECT * FROM personal_records").fetchall()]
    conn.close()

    # Get demographics from main DB
    main = sqlite3.connect('data/boxbunny_main.db')
    main.row_factory = sqlite3.Row
    user = dict(main.execute("SELECT * FROM users WHERE username=?", (username,)).fetchone())
    main.close()

    # Build records summary
    records_html = ""
    for r in records[:4]:  # Show up to 4 records
        records_html += (
            f'<div style="background:#111;padding:6px 10px;border-radius:6px;margin:2px 0;font-size:12px">'
            f'<span style="color:#FFC107">{r["metric"].replace("_"," ").title()}</span>: '
            f'<b style="color:white">{r["value"]}</b>'
            f'</div>'
        )

    # XP progress bar
    xp_pct = min(100, int(xp.get('level_progress', 0) * 100)) if 'level_progress' in xp else 50

    cards_html += f"""
    <div style="background:#1A1A1A;padding:20px;border-radius:12px;margin:12px 0;
                font-family:sans-serif;color:white;border-left:4px solid #00E676">
        <div style="display:flex;justify-content:space-between;align-items:center">
            <h3 style="color:#00E676;margin:0">{user['display_name']}</h3>
            <span style="background:#00E676;color:#0D0D0D;padding:4px 12px;border-radius:12px;
                         font-weight:bold;font-size:13px">{xp['current_rank']}</span>
        </div>
        <p style="color:#9E9E9E;margin:4px 0 0 0;font-size:13px">
            {user.get('gender','?').title()}, Age {user.get('age','?')} |
            {user.get('height_cm','?')}cm, {user.get('weight_kg','?')}kg |
            {user['level'].title()} | {user.get('stance','orthodox').title()}
        </p>

        <div style="display:flex;gap:12px;margin-top:16px">
            <div style="background:#0D0D0D;padding:14px;border-radius:8px;flex:1;text-align:center">
                <div style="font-size:28px;font-weight:bold;color:#00E676">{xp['total_xp']}</div>
                <div style="color:#9E9E9E;font-size:11px">TOTAL XP</div>
            </div>
            <div style="background:#0D0D0D;padding:14px;border-radius:8px;flex:1;text-align:center">
                <div style="font-size:28px;font-weight:bold">{sessions}</div>
                <div style="color:#9E9E9E;font-size:11px">SESSIONS</div>
            </div>
            <div style="background:#0D0D0D;padding:14px;border-radius:8px;flex:1;text-align:center">
                <div style="font-size:28px;font-weight:bold;color:#FF5722">{streak['current_streak']}</div>
                <div style="color:#9E9E9E;font-size:11px">DAY STREAK</div>
            </div>
            <div style="background:#0D0D0D;padding:14px;border-radius:8px;flex:1;text-align:center">
                <div style="font-size:28px;font-weight:bold;color:#FFC107">{achievements}</div>
                <div style="color:#9E9E9E;font-size:11px">ACHIEVEMENTS</div>
            </div>
        </div>

        <div style="margin-top:12px">
            <div style="color:#9E9E9E;font-size:11px;margin-bottom:4px">PERSONAL RECORDS</div>
            <div style="display:flex;gap:6px;flex-wrap:wrap">{records_html}</div>
        </div>
    </div>
    """

display(HTML(f"""
<div style="max-width:700px">
    <h2 style="color:#E0E0E0;font-family:sans-serif;margin-bottom:4px">Demo User Profiles</h2>
    <p style="color:#9E9E9E;font-family:sans-serif;font-size:13px;margin-top:0">
        Data pulled live from per-user SQLite databases in data/users/
    </p>
    {cards_html}
</div>
"""))

---
## 28. Benchmark Percentile Test
Test the population benchmark engine with each demo user's profile. The engine computes percentile rankings by comparing performance metrics against population norms segmented by age, gender, and skill level.

Results show where each user falls relative to the general population for their demographic bracket.

In [ ]:
import sys
sys.path.insert(0, 'src/boxbunny_dashboard')
from boxbunny_dashboard.benchmarks import BenchmarkEngine
from IPython.display import HTML, display

engine = BenchmarkEngine()

# Define test cases matching our demo users
test_cases = [
    {
        "name": "Alex",
        "subtitle": "Male, 22, Beginner",
        "stats": {"avg_reaction_ms": 340, "punches_per_minute": 47, "total_punches": 95},
        "age": 22, "gender": "male", "level": "beginner",
    },
    {
        "name": "Maria",
        "subtitle": "Female, 28, Intermediate",
        "stats": {"avg_reaction_ms": 280, "punches_per_minute": 65, "defense_rate": 0.65},
        "age": 28, "gender": "female", "level": "intermediate",
    },
    {
        "name": "Jake",
        "subtitle": "Male, 31, Advanced",
        "stats": {"avg_reaction_ms": 220, "punches_per_minute": 90, "defense_rate": 0.80},
        "age": 31, "gender": "male", "level": "advanced",
    },
]

all_html = ""
for tc in test_cases:
    results = engine.get_all_percentiles(
        tc["stats"], age=tc["age"], gender=tc["gender"], level=tc["level"]
    )

    rows_html = ""
    for metric, data in results.items():
        pct = data.get("percentile", 50)
        tier = data.get("tier", "Unknown")
        comparison = data.get("comparison", "")

        # Colour the bar based on percentile
        if pct >= 75:
            bar_color = "#00E676"
        elif pct >= 50:
            bar_color = "#FFC107"
        elif pct >= 25:
            bar_color = "#FF9800"
        else:
            bar_color = "#FF5722"

        metric_label = metric.replace("_", " ").title()
        rows_html += f"""
        <tr>
            <td style="padding:8px;color:#E0E0E0;font-size:13px;white-space:nowrap">{metric_label}</td>
            <td style="padding:8px;width:200px">
                <div style="background:#333;border-radius:4px;height:20px;position:relative">
                    <div style="background:{bar_color};border-radius:4px;height:20px;width:{pct}%;
                                display:flex;align-items:center;justify-content:center;
                                font-size:11px;font-weight:bold;color:#0D0D0D;min-width:30px">
                        {pct}%
                    </div>
                </div>
            </td>
            <td style="padding:8px;color:{bar_color};font-weight:bold;font-size:13px">{tier}</td>
            <td style="padding:8px;color:#9E9E9E;font-size:12px">{comparison}</td>
        </tr>
        """

    all_html += f"""
    <div style="background:#1A1A1A;padding:16px;border-radius:12px;margin:12px 0;font-family:sans-serif">
        <h3 style="color:#00E676;margin:0 0 4px 0">{tc['name']}</h3>
        <p style="color:#9E9E9E;margin:0 0 12px 0;font-size:13px">{tc['subtitle']}</p>
        <table style="width:100%;border-collapse:collapse">
            <tr style="border-bottom:1px solid #333">
                <th style="text-align:left;padding:6px;color:#9E9E9E;font-size:11px">METRIC</th>
                <th style="text-align:left;padding:6px;color:#9E9E9E;font-size:11px">PERCENTILE</th>
                <th style="text-align:left;padding:6px;color:#9E9E9E;font-size:11px">TIER</th>
                <th style="text-align:left;padding:6px;color:#9E9E9E;font-size:11px">COMPARISON</th>
            </tr>
            {rows_html}
        </table>
    </div>
    """

display(HTML(f"""
<div style="max-width:800px">
    <h2 style="color:#E0E0E0;font-family:sans-serif;margin-bottom:4px">Population Benchmark Results</h2>
    <p style="color:#9E9E9E;font-family:sans-serif;font-size:13px;margin-top:0">
        Percentile rankings computed against age/gender/level population norms
    </p>
    {all_html}
</div>
"""))